[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/12_gradient_boosting/02_gradient_boosting_from_scratch.ipynb)

# Bài 02 — Gradient Boosting from scratch

**🎯 Mục tiêu**
- Hiểu Gradient Boosting = **gradient descent trong không gian hàm**: mỗi cây xấp xỉ **negative gradient** (pseudo-residual) của loss.
- Cài đặt GB hồi quy (MSE → residual) và xem từng bước; vai trò **learning rate** (shrinkage) và số cây.
- Đổi loss: MAE/Huber (robust với outlier); phân loại với **log-loss** (pseudo-residual $y - p$).
- Stochastic GB (`subsample`) và **early stopping**; đối chiếu với scikit-learn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/12_gradient_boosting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from ml_course.datasets import make_regression_1d
from ml_course.utils import sigmoid
from ml_course.visualization import set_style

set_style()

## 1. Từ gradient descent tham số sang gradient descent hàm

Gradient descent quen thuộc: $\theta \leftarrow \theta - \eta\,\nabla_\theta L$. Gradient Boosting coi **giá trị dự đoán** $F(x_i)$ tại mỗi điểm train là "tham số":

$$r_i = -\left.\frac{\partial L(y_i, F)}{\partial F}\right|_{F = F_{m-1}(x_i)} \qquad\text{(pseudo-residual)}$$

Nhưng ta cần một **hàm** dự đoán được cho điểm mới → fit một cây hồi quy $h_m$ vào các cặp $(x_i, r_i)$, rồi bước một bước nhỏ:

$$F_m(x) = F_{m-1}(x) + \eta\, h_m(x)$$

| Loss $L(y, F)$ | Pseudo-residual $-\partial L/\partial F$ |
|---|---|
| MSE $\frac12(y - F)^2$ | $y - F$ — **chính là residual** |
| MAE $\lvert y - F\rvert$ | $\operatorname{sign}(y - F)$ |
| Log-loss (nhị phân, $F$ = log-odds) | $y - \sigma(F)$ |

## 2. GB hồi quy với MSE — từng bước

In [ ]:
class GradientBoostingMSE:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=2, subsample=1.0, seed=0):
        self.n_estimators, self.learning_rate, self.max_depth = n_estimators, learning_rate, max_depth
        self.subsample, self.seed = subsample, seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        self.init_ = y.mean()                                  # F_0 = hằng số tối ưu cho MSE
        F = np.full(len(y), self.init_)
        self.trees_ = []
        for _ in range(self.n_estimators):
            residual = y - F                                   # negative gradient của MSE
            idx = rng.choice(len(y), int(self.subsample * len(y)), replace=False) if self.subsample < 1 else slice(None)
            tree = DecisionTreeRegressor(criterion="friedman_mse", max_depth=self.max_depth, random_state=0).fit(X[idx], residual[idx])
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def staged_predict(self, X):
        F = np.full(len(X), self.init_)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
            yield F

    def predict(self, X):
        return self.init_ + self.learning_rate * sum(t.predict(X) for t in self.trees_)


X, y = make_regression_1d(n=200, kind="sin", noise=0.5, seed=0)
grid = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
gb = GradientBoostingMSE(n_estimators=100, learning_rate=0.3, max_depth=1).fit(X, y)
stages = list(gb.staged_predict(grid))
F_train = [y.mean()] + list(gb.staged_predict(X))

fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for col, m in enumerate([1, 3, 10, 100]):
    axes[0, col].scatter(X, y, s=8, alpha=0.5)
    axes[0, col].plot(grid, stages[m - 1], "r", lw=2)
    axes[0, col].set_title(f"F_{m}(x) sau {m} cây")
    residual = y - F_train[m - 1]
    axes[1, col].scatter(X, residual, s=8, alpha=0.5, color="gray")
    axes[1, col].plot(grid, gb.trees_[m - 1].predict(grid), "g", lw=2)
    axes[1, col].set_title(f"residual trước cây {m} + cây {m} fit vào nó")
plt.tight_layout()
plt.show()

Hàng dưới: mỗi cây (stump, xanh lá) chỉ học **phần còn sai** (residual) của tổng các cây trước. Residual nhỏ dần → cây sau càng "tinh chỉnh" chi tiết.

## 3. Learning rate × số cây

Learning rate $\eta$ nhỏ (shrinkage) → mỗi cây đóng góp ít → cần **nhiều cây hơn**, nhưng thường **tổng quát hoá tốt hơn** (Friedman, 2001). Hai hyperparameter này luôn đi cùng nhau:

In [ ]:
X_val, y_val = make_regression_1d(n=1000, kind="sin", noise=0.5, seed=1)
fig, ax = plt.subplots(figsize=(9, 4.5))
for lr in [1.0, 0.3, 0.1, 0.03]:
    model = GradientBoostingMSE(n_estimators=600, learning_rate=lr, max_depth=2).fit(X, y)
    val_mse = [mean_squared_error(y_val, F) for F in model.staged_predict(X_val)]
    best = int(np.argmin(val_mse))
    ax.plot(range(1, 601), val_mse, label=f"η = {lr}: tốt nhất {val_mse[best]:.3f} tại {best + 1} cây")
ax.axhline(0.25, color="k", ls=":", label="nhiễu σ² = 0.25 (giới hạn dưới)")
ax.set_xscale("log")
ax.set_ylim(0.2, 1.0)
ax.set_xlabel("số cây")
ax.set_ylabel("MSE validation")
ax.legend(fontsize=8)
ax.set_title("η lớn: nhanh rồi overfit; η nhỏ: chậm nhưng ổn định")
plt.show()

Khác với Random Forest, **thêm cây có thể overfit** — đường validation đi lên sau điểm tốt nhất (rõ nhất với $\eta$ lớn). Cần chọn số cây bằng validation / **early stopping**.

## 4. Đối chiếu với scikit-learn

In [ ]:
ours = GradientBoostingMSE(n_estimators=200, learning_rate=0.1, max_depth=3).fit(X, y)
sk = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
print(f"MSE validation: scratch = {mean_squared_error(y_val, ours.predict(X_val)):.4f} | sklearn = {mean_squared_error(y_val, sk.predict(X_val)):.4f}")
print("dự đoán gần như trùng nhau:", np.allclose(ours.predict(X_val), sk.predict(X_val), atol=1e-6))

## 5. Đổi loss: robust với outlier

Với MSE, residual của outlier rất lớn → các cây dồn sức "kéo" về outlier. MAE dùng $\operatorname{sign}(y - F)$ → mỗi điểm đóng góp như nhau dù sai bao nhiêu; **Huber** kết hợp cả hai.

In [ ]:
rng = np.random.default_rng(3)
y_dirty = y.copy()
out = rng.choice(len(y), 15, replace=False)
y_dirty[out] += rng.choice([-8, 8], 15)                    # 7.5% outlier
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(X, y_dirty, s=8, alpha=0.5)
for loss, color in [("squared_error", "tab:red"), ("absolute_error", "tab:green"), ("huber", "tab:purple")]:
    m = GradientBoostingRegressor(loss=loss, n_estimators=300, learning_rate=0.05, max_depth=2, random_state=0).fit(X, y_dirty)
    ax.plot(grid, m.predict(grid), color=color, lw=2,
            label=f"{loss}: MSE so với dữ liệu sạch (validation) = {mean_squared_error(y_val, m.predict(X_val)):.3f}")
ax.set_ylim(-6, 6)
ax.legend(fontsize=8)
ax.set_title("Outlier kéo lệch GB với MSE; MAE/Huber robust hơn")
plt.show()

## 6. Phân loại với log-loss

$F(x)$ là **log-odds**, $p = \sigma(F)$. Pseudo-residual $= y - p$. Khởi tạo $F_0 = \log\frac{\bar y}{1 - \bar y}$. Giá trị lá: sklearn dùng một **bước Newton** $\frac{\sum r_i}{\sum p_i(1 - p_i)}$ thay cho trung bình residual — hội tụ nhanh hơn. Ở đây ta cài đặt bước Newton đó:

In [ ]:
class GradientBoostingBinary:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators, self.learning_rate, self.max_depth = n_estimators, learning_rate, max_depth

    def fit(self, X, y):
        p0 = y.mean()
        self.init_ = np.log(p0 / (1 - p0))
        F = np.full(len(y), self.init_)
        self.trees_ = []
        for _ in range(self.n_estimators):
            p = sigmoid(F)
            residual = y - p                                            # negative gradient của log-loss
            tree = DecisionTreeRegressor(criterion="friedman_mse", max_depth=self.max_depth, random_state=0).fit(X, residual)
            leaf = tree.apply(X)                                        # lá của mỗi mẫu
            for node in np.unique(leaf):                               # bước Newton cho từng lá
                m = leaf == node
                tree.tree_.value[node, 0, 0] = residual[m].sum() / max(np.sum(p[m] * (1 - p[m])), 1e-12)
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def predict_proba(self, X):
        F = self.init_ + self.learning_rate * sum(t.predict(X) for t in self.trees_)
        return sigmoid(F)


from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=4000, seed=5)
Xc = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
yc = churn["churn"].to_numpy().astype(float)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)
ours_c = GradientBoostingBinary(n_estimators=150, learning_rate=0.1, max_depth=3).fit(Xc_tr, yc_tr)
sk_c = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=0).fit(Xc_tr, yc_tr)
print(f"log-loss test: scratch = {log_loss(yc_te, ours_c.predict_proba(Xc_te)):.4f} | sklearn = {log_loss(yc_te, sk_c.predict_proba(Xc_te)[:, 1]):.4f}")

> ⚠️ Ghi đè `tree_.value` là thủ thuật minh hoạ (sklearn làm điều tương tự bên trong). Phiên bản thuần NumPy: `ml_course.algorithms.GradientBoostingClassifierScratch`.

## 7. Stochastic GB và early stopping

- `subsample < 1`: mỗi cây chỉ thấy một phần dữ liệu (không hoàn lại) → thêm ngẫu nhiên, giảm variance, nhanh hơn (Friedman, 2002).
- **Early stopping**: dừng khi điểm validation không cải thiện sau `n_iter_no_change` vòng.

In [ ]:
for sub in [1.0, 0.5]:
    m = GradientBoostingClassifier(n_estimators=2000, learning_rate=0.05, max_depth=3, subsample=sub,
                                   validation_fraction=0.2, n_iter_no_change=30, random_state=0).fit(Xc_tr, yc_tr)
    print(f"subsample = {sub}: dừng sau {m.n_estimators_} cây | log-loss test = {log_loss(yc_te, m.predict_proba(Xc_te)[:, 1]):.4f}")

## 🧠 Kiểm tra nhanh

1. Với MSE, pseudo-residual là gì? Với log-loss?
2. Vì sao Gradient Boosting cần fit **cây** vào pseudo-residual thay vì cập nhật trực tiếp $F(x_i)$?
3. Giảm learning rate 10 lần thì nên làm gì với số cây?
4. Vì sao GB với MAE robust hơn với outlier?

## 📌 Tóm tắt

- GB = gradient descent trong không gian hàm: cây $h_m$ xấp xỉ negative gradient, $F_m = F_{m-1} + \eta h_m$.
- MSE → residual; MAE → dấu; log-loss → $y - p$ (+ bước Newton ở lá).
- $\eta$ nhỏ + nhiều cây + early stopping; `subsample` thêm ngẫu nhiên.
- Thêm cây **có thể** overfit — khác Random Forest.

## 📚 Tài liệu tham khảo

- Friedman (2001) — *Greedy Function Approximation: A Gradient Boosting Machine*
- Friedman (2002) — *Stochastic Gradient Boosting*
- ESL §10.10–10.12